# Patient Data Ingestion

Reads `patient_segmentation_dataset.csv`, maps the text columns to their lookup-table IDs, and bulk-inserts all 2,000 rows into the `patients` table.

**Run the SQL scripts first (in order):**
1. `01_create_database.sql`
2. `02_create_lookup_tables.sql`
3. `03_create_patients_table.sql`
4. `04_seed_lookup_tables.sql`  ← seeds states, insurance_types, conditions
5. ✅ This notebook  ← loads the 2,000 patient rows

**Requirement:** `pip install pandas mysql-connector-python`

## 1 · Config — edit only this cell

In [ ]:
import pathlib

# --- database connection ---
DB_HOST     = "localhost"
DB_PORT     = 3306
DB_NAME     = "patient_data"
DB_USER     = "root"        # change if you use a different MySQL user
DB_PASSWORD = ""            # fill in your MySQL root password

# --- CSV path (relative to wherever you launch Jupyter from) ---
CSV_PATH = pathlib.Path("patient_segmentation_dataset.csv")

## 2 · Load CSV

In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
df.head(3)

## 3 · Connect to MySQL and fetch lookup maps

In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
)
cur = conn.cursor()
print("Connected:", conn.get_server_info())

In [ ]:
# Build {text_value: integer_id} dicts by reading the lookup tables.
# This way the mapping is always in sync with whatever IDs MySQL assigned
# — we never hard-code a number.

def fetch_map(sql, key_idx, val_idx):
    cur.execute(sql)
    return {row[key_idx]: row[val_idx] for row in cur.fetchall()}

state_map     = fetch_map("SELECT state_code, state_id           FROM states",         0, 1)
insurance_map = fetch_map("SELECT type_name,  insurance_type_id  FROM insurance_types", 0, 1)
condition_map = fetch_map("SELECT condition_name, condition_id   FROM conditions",      0, 1)

print("States:",     state_map)
print("Insurance:",  insurance_map)
print("Conditions:", condition_map)

## 4 · Transform — map text columns to IDs

In [ ]:
# Map text → integer FK
df["state_id"]          = df["State"].map(state_map)
df["insurance_type_id"] = df["Insurance_Type"].map(insurance_map)

# CSV value "None" → Python None → SQL NULL
# All other condition names look up their ID normally
df["condition_id"] = df["Primary_Condition"].apply(
    lambda v: condition_map.get(v) if v != "None" else None
)

# City: literal string 'Unknown' → None → SQL NULL
df["city_clean"] = df["City"].where(df["City"] != "Unknown", other=None)

# Sanity check — unmapped state or insurance values would show as NaN
unmapped = {
    "state":     df["state_id"].isna().sum(),
    "insurance": df["insurance_type_id"].isna().sum(),
}
print("Unmapped counts (both must be 0 before continuing):", unmapped)

# Convert ID columns to pandas nullable int so they don't become floats
for col in ("state_id", "insurance_type_id", "condition_id"):
    df[col] = df[col].astype("Int64")

df[["PatientID", "State", "state_id",
    "Insurance_Type", "insurance_type_id",
    "Primary_Condition", "condition_id",
    "City", "city_clean"]].head(5)

## 5 · Insert all 2,000 rows into `patients`

In [ ]:
INSERT_SQL = """
INSERT INTO patients (
    patient_id, age, gender, state_id, city,
    height_cm, weight_kg, bmi,
    insurance_type_id, condition_id,
    num_chronic_conditions, annual_visits,
    avg_billing_amount, last_visit_date,
    days_since_last_visit, preventive_care_flag
) VALUES (
    %s, %s, %s, %s, %s,
    %s, %s, %s,
    %s, %s,
    %s, %s,
    %s, %s,
    %s, %s
)
"""

def nullable_int(v):
    return None if pd.isna(v) else int(v)

def nullable_float(v):
    return None if pd.isna(v) else float(v)

def to_tuple(r):
    return (
        r["PatientID"],
        nullable_int(r["Age"]),
        r["Gender"],
        nullable_int(r["state_id"]),
        r["city_clean"],
        nullable_int(r["Height_cm"]),
        nullable_int(r["Weight_kg"]),
        nullable_float(r["BMI"]),
        nullable_int(r["insurance_type_id"]),
        nullable_int(r["condition_id"]),
        nullable_int(r["Num_Chronic_Conditions"]),
        nullable_int(r["Annual_Visits"]),
        nullable_float(r["Avg_Billing_Amount"]),
        r["Last_Visit_Date"],
        nullable_int(r["Days_Since_Last_Visit"]),
        bool(r["Preventive_Care_Flag"]),
    )

rows = [to_tuple(r) for _, r in df.iterrows()]

cur.executemany(INSERT_SQL, rows)
conn.commit()
print(f"Inserted {cur.rowcount:,} rows into patients")

## 6 · Verify

In [ ]:
cur.execute("SELECT COUNT(*) FROM patients")
print(f"Total rows in patients: {cur.fetchone()[0]:,}  (expected 2,000)")

print("\nBreakdown by state:")
cur.execute("""
    SELECT s.state_code, COUNT(*) AS n
    FROM patients p
    JOIN states s USING (state_id)
    GROUP BY s.state_code
    ORDER BY n DESC
""")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]}")

print("\nBreakdown by insurance type:")
cur.execute("""
    SELECT i.type_name, COUNT(*) AS n
    FROM patients p
    JOIN insurance_types i USING (insurance_type_id)
    GROUP BY i.type_name
    ORDER BY n DESC
""")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]}")

In [ ]:
cur.close()
conn.close()
print("Connection closed.")